## Setup and Import

In [1]:
import os, warnings
warnings.filterwarnings('ignore')

CFG = dict(
    SEED          = 42,
    N_FOLDS       = 2,
    EPOCHS_DEB    = 2,
    BS_DEB        = 4,
    GRAD_ACCUM_DEB  = 1,
    MAX_LEN_DEB   = 512,
    LR_DEB        = 1e-5,
    MARGIN        = 0.5,
    USE_BF16      = True,
    DATA_DIR      = '/kaggle/input/competitions/smart-mcq-solver-challenge',
    OUTPUT_DIR    = '/kaggle/working',
    ARTIFACT_DIR  = '/kaggle/working/artifacts',  
    DEBERTA_MODEL = 'microsoft/deberta-v3-large'
)

os.makedirs(CFG['OUTPUT_DIR'], exist_ok=True)
os.makedirs(CFG['ARTIFACT_DIR'], exist_ok=True)
OPTS = ['A', 'B', 'C', 'D', 'E']

In [2]:
import os, sys, json, math, random, time, gc, re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import autocast as _autocast, GradScaler as _GradScaler

from sklearn.metrics import f1_score, accuracy_score


def seed_all(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_all(CFG['SEED'])

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if DEVICE.type == 'cuda':
    p = torch.cuda.get_device_properties(0)
    print(f"  GPU: {p.name}, {p.total_memory/1e9:.1f} GB, compute {p.major}.{p.minor}")

if CFG['USE_BF16'] and DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported():
    AMP_DTYPE = torch.bfloat16
elif DEVICE.type == 'cuda':
    AMP_DTYPE = torch.float16
else:
    AMP_DTYPE = torch.float32
USE_SCALER = (AMP_DTYPE == torch.float16)
print(f"AMP dtype: {AMP_DTYPE} | GradScaler: {USE_SCALER}")

if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    torch.set_float32_matmul_precision('high')

def autocast():
    return _autocast(device_type='cuda', dtype=AMP_DTYPE, enabled=(DEVICE.type == 'cuda'))

def make_scaler():
    return _GradScaler('cuda', enabled=USE_SCALER)

Device: cpu
AMP dtype: torch.float32 | GradScaler: False


In [3]:
train = pd.read_csv(f"{CFG['DATA_DIR']}/train.csv")
test  = pd.read_csv(f"{CFG['DATA_DIR']}/test.csv")
y_idx = np.array([OPTS.index(a) for a in train['answer']])

In [4]:
import re
from sklearn.model_selection import GroupKFold

_BOILERPLATE = [
    r"pick the best possible answer:?",
    r"\bcarefully\b\.?",
    r"among the listed options\.?",
    r"choose the (correct|best) (option|answer|response)\.?",
    r"select the (best|correct|most appropriate) (option|answer|response)\.?",
    r"determine the correct option:?",
    r"identify the correct statement:?",
    r"based on the given context\.?",
    r"which of the following( statements)?( is true)?",
]
_WS = re.compile(r'\s+')

def normalize_text(text):
    s = str(text)
    for pat in _BOILERPLATE:
        s = re.sub(pat, ' ', s, flags=re.IGNORECASE)
    return _WS.sub(' ', s).strip()

for df in (train, test):
    df['prompt_n'] = df['prompt'].map(normalize_text)
    for o in OPTS:
        df[f'{o}_n'] = df[o].map(normalize_text)


gkf = GroupKFold(n_splits=CFG['N_FOLDS'])
train['fold'] = -1
for fold_id, (_, va_idx) in enumerate(gkf.split(train, groups=train['prompt_n'])):
    train.loc[train.index[va_idx], 'fold'] = fold_id

assert (train['fold'] >= 0).all()
_dup = train.groupby('prompt_n')['fold'].nunique()
assert _dup.max() == 1, "Leak: a prompt spans multiple folds"
print("Folds:\n", train['fold'].value_counts().sort_index().to_string())

Folds:
 fold
0    1000
1    1000


In [5]:
def map_at_3(scores_2d, y_idx):
    scores_2d = np.asarray(scores_2d).reshape(-1, 5)
    y_idx = np.asarray(y_idx)
    order = (-scores_2d).argsort(axis=1)
    total = 0.0
    for i in range(len(scores_2d)):
        top3 = list(order[i, :3])
        if y_idx[i] in top3:
            total += 1.0 / (top3.index(y_idx[i]) + 1)
    return total / len(scores_2d)

def all_metrics(scores_2d, y_idx):
    scores_2d = np.asarray(scores_2d).reshape(-1, 5)
    pred = scores_2d.argmax(axis=1)
    return dict(map3=map_at_3(scores_2d, y_idx),
                acc=accuracy_score(y_idx, pred),
                f1=f1_score(y_idx, pred, average='macro'))

In [6]:
def pairwise_margin_loss(scores, correct, margin=0.5):

    B = scores.size(0)
    pos = scores[torch.arange(B), correct].unsqueeze(1).expand(-1, 5)
    target = torch.ones_like(scores)
    loss = F.margin_ranking_loss(pos, scores, target, margin=margin, reduction='none')
    mask = torch.ones_like(loss)
    mask[torch.arange(B), correct] = 0
    return (loss * mask).sum() / mask.sum().clamp(min=1)

### Tracking (Weights & Biases)

In [ ]:
import os
try:
    import wandb
except ImportError:
    wandb = None


USE_WANDB     = True
RUN_VERSION   = 1
RUN_NOTES     = "run_1, n_folds=2, epoch = 1"      
WANDB_PROJECT = "24f1000134-t22026"

if USE_WANDB and wandb is not None:
    try:
        from kaggle_secrets import UserSecretsClient
        wandb.login(key=UserSecretsClient().get_secret("wandb_api"))
        print("W&B connected.")
    except Exception as e:
        print(f"W&B login skipped ({e}); set Kaggle secret 'wandb_api' to enable.")
        USE_WANDB = False
elif wandb is None:
    USE_WANDB = False
    print("wandb not installed; logging disabled.")

def wandb_init(model_tag, fold):
    """One W&B run per (model, fold). Run name encodes the version = nth run."""
    if not (USE_WANDB and wandb is not None):
        return None
    return wandb.init(
        project=WANDB_PROJECT,
        name=f"{model_tag}_v{RUN_VERSION}_fold{fold}",
        group=f"{model_tag}_v{RUN_VERSION}",          
        job_type="train",
        notes=RUN_NOTES,                           
        tags=[model_tag, f"v{RUN_VERSION}"],
        config={**CFG, "run_version": RUN_VERSION,
                "run_notes": RUN_NOTES, "model": model_tag, "fold": fold},
        reinit=True,
    )

def wlog(metrics, step=None):
    if USE_WANDB and wandb is not None and wandb.run is not None:
        wandb.log(metrics, step=step)

def wandb_finish():
    if USE_WANDB and wandb is not None and wandb.run is not None:
        wandb.finish()

### Uploading Model to Kagglehub

In [ ]:
try:
    import kagglehub
except ImportError:
    kagglehub = None


KAGGLE_USERNAME = "Kamal134134"
PUSH_TO_HUB     = True
HUB_FRAMEWORK   = "pyTorch"

HUB_HANDLES = {
    "deberta": f"{KAGGLE_USERNAME}/smart-mcq-deberta/{HUB_FRAMEWORK}/baseline"
}

def push_model(model_tag, folder, notes=""):

    if not PUSH_TO_HUB or kagglehub is None:
        print(f"[hub] upload skipped for '{model_tag}' (PUSH_TO_HUB=False).")
        return
    handle = HUB_HANDLES[model_tag]
    note = notes or f"v{RUN_VERSION}: {RUN_NOTES}"
    print(f"[hub] uploading '{model_tag}' -> {handle}")
    kagglehub.model_upload(handle, folder, version_notes=note)
    print(f"[hub] done. Inference notebooks can now kagglehub.model_download('{handle}').")

## 1. HuggingFace auth

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF token loaded.")
else:
    print("No HF token found — public models will still download, gated ones won't.")

## 2. Model + dataset — a DeBERTa cross-encoder

In [7]:
from transformers import AutoTokenizer, AutoModel

class DebertaScorer(nn.Module):
    
    def __init__(self, model_name):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name, token=HF_TOKEN or None)
        h = self.backbone.config.hidden_size
        self.head = nn.Sequential(
            nn.Dropout(0.1), nn.Linear(h, h // 2), nn.GELU(),
            nn.Dropout(0.1), nn.Linear(h // 2, 1),
        )

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.head(cls).squeeze(-1)


class DebertaDataset(Dataset):
    def __init__(self, df, tokenizer, max_len, shuffle_opts=False, has_label=True, seed=0):
        self.df = df.reset_index(drop=True)
        self.tok = tokenizer
        self.max_len = max_len
        self.shuffle = shuffle_opts
        self.has_label = has_label
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        prompt = r['prompt_n']
        options = [r[f'{o}_n'] for o in OPTS]
        correct = OPTS.index(r['answer']) if self.has_label else -1
        if self.shuffle:
            perm = self.rng.permutation(5)
            options = [options[i] for i in perm]
            if self.has_label:
                correct = int(np.where(perm == correct)[0][0])
        enc = self.tok([prompt] * 5, options, truncation=True,
                       padding='max_length', max_length=self.max_len,
                       return_tensors='pt')
        return dict(input_ids=enc['input_ids'],
                    attention_mask=enc['attention_mask'],
                    correct=torch.tensor(correct, dtype=torch.long),
                    id=int(r['id']))

## 3. Cross-validated training

In [8]:
MODEL_TAG  = 'deberta'
UPLOAD_DIR = f"{CFG['ARTIFACT_DIR']}/{MODEL_TAG}"
os.makedirs(UPLOAD_DIR, exist_ok=True)


@torch.no_grad()
def predict_deberta(model, dl):
    model.eval(); out = []
    for batch in dl:
        B = batch['input_ids'].size(0)
        ids = batch['input_ids'].view(B * 5, -1).to(DEVICE)
        mask = batch['attention_mask'].view(B * 5, -1).to(DEVICE)
        with autocast():
            s = model(ids, mask).view(B, 5)
        out.append(s.cpu().float().numpy())
    return np.concatenate(out)


def train_deberta_cv(train_df, test_df, n_folds=CFG['N_FOLDS'], save_dir=UPLOAD_DIR):
    oof = np.zeros((len(train_df), 5), dtype=np.float32)
    te_pred = np.zeros((len(test_df), 5), dtype=np.float32)
    y = np.array([OPTS.index(a) for a in train_df['answer']])
    tok = AutoTokenizer.from_pretrained(CFG['DEBERTA_MODEL'], token=HF_TOKEN or None)
    te_dl = DataLoader(DebertaDataset(test_df, tok, CFG['MAX_LEN_DEB'], has_label=False),
                       batch_size=CFG['BS_DEB'], shuffle=False)

    for fold in range(n_folds):
        run = wandb_init(MODEL_TAG, fold)
        va_idx = np.where(train_df['fold'].values == fold)[0]
        tr_idx = np.where(train_df['fold'].values != fold)[0]
        tr_dl = DataLoader(DebertaDataset(train_df.iloc[tr_idx], tok, CFG['MAX_LEN_DEB'],
                                          shuffle_opts=True, seed=fold),
                           batch_size=CFG['BS_DEB'], shuffle=True)
        va_dl = DataLoader(DebertaDataset(train_df.iloc[va_idx], tok, CFG['MAX_LEN_DEB']),
                           batch_size=CFG['BS_DEB'], shuffle=False)

        model = DebertaScorer(CFG['DEBERTA_MODEL']).to(DEVICE)
        optim = AdamW(model.parameters(), lr=CFG['LR_DEB'], weight_decay=0.01)
        accum = CFG['GRAD_ACCUM_DEB']
        sched = CosineAnnealingLR(optim, T_max=CFG['EPOCHS_DEB'] * max(len(tr_dl)//accum, 1))
        scaler = make_scaler()

        best_map3, best_va, best_te = -1, None, None
        for epoch in range(CFG['EPOCHS_DEB']):
            model.train(); optim.zero_grad(); running, seen = 0.0, 0
            for step, batch in enumerate(tr_dl):
                B = batch['input_ids'].size(0)
                ids = batch['input_ids'].view(B * 5, -1).to(DEVICE)
                mask = batch['attention_mask'].view(B * 5, -1).to(DEVICE)
                correct = batch['correct'].to(DEVICE)
                with autocast():
                    scores = model(ids, mask).view(B, 5)
                    loss = pairwise_margin_loss(scores, correct, CFG['MARGIN']) / accum
                if torch.isfinite(loss):
                    scaler.scale(loss).backward()
                if (step + 1) % accum == 0:
                    if USE_SCALER: scaler.unscale_(optim)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optim); scaler.update(); sched.step(); optim.zero_grad()
                running += loss.item() * accum * B; seen += B

            va_scores = predict_deberta(model, va_dl)
            m = all_metrics(va_scores, y[va_idx])
            wlog({'train_loss': running / max(seen, 1), 'val_map3': m['map3'],
                  'val_acc': m['acc'], 'val_f1': m['f1'], 'epoch': epoch}, step=epoch)
            print(f"  fold {fold} ep {epoch}: MAP@3={m['map3']:.4f} acc={m['acc']:.4f}")
            if m['map3'] > best_map3:
                best_map3, best_va = m['map3'], va_scores
                best_te = predict_deberta(model, te_dl)
                torch.save(model.state_dict(), f"{save_dir}/best_deb_fold{fold}.pth")

        wlog({'best_val_map3': best_map3})
        oof[va_idx] = best_va; te_pred += best_te / n_folds
        wandb_finish()
        del model; gc.collect(); torch.cuda.empty_cache()
    return oof, te_pred


oof_deb, test_deb = train_deberta_cv(train, test)
print(f"\nDeBERTa OOF MAP@3: {map_at_3(oof_deb, y_idx):.4f}")
np.save(f"{UPLOAD_DIR}/oof_deb.npy",  oof_deb)
np.save(f"{UPLOAD_DIR}/test_deb.npy", test_deb)
push_model(MODEL_TAG, UPLOAD_DIR,
           notes=f"DeBERTa v{RUN_VERSION} | OOF MAP@3={map_at_3(oof_deb, y_idx):.4f} | {RUN_NOTES}")

NameError: name 'HF_TOKEN' is not defined